# TP2 — Descente de gradient : loi d’Abrams (génie civil)

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/earthai-tech/lab-courses/blob/main/inphb/prepa/notebooks/PREPA_TP2_descente_gradient.ipynb)
Objectif : coder la descente de gradient, étudier le pas η, le conditionnement et la standardisation.

> Données : dossier `PREPA_TP_donnees`. Règle du module : **numpy uniquement**, pas de bibliothèque d’apprentissage.

In [ ]:
# Configuration : Colab ou Jupyter local. Exécutez cette cellule en premier.
import os, shutil, urllib.request
DONNEES = ['beton_resistance.csv']
URL = 'https://raw.githubusercontent.com/earthai-tech/lab-courses/main/inphb/prepa/data/'
for f in DONNEES:
    if os.path.exists(f):
        continue
    if os.path.exists(os.path.join('..', 'data', f)):
        shutil.copy(os.path.join('..', 'data', f), f)   # kit hors ligne / dépôt cloné
    else:
        urllib.request.urlretrieve(URL + f, f)            # Colab : téléchargement
    print('OK', f)


In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
d = pd.read_csv('beton_resistance.csv')
x = d['rapport_E_C'].values; y = np.log(d['resistance_MPa'].values); n = len(y)
X = np.c_[np.ones(n), x]
theta_exact = np.linalg.solve(X.T @ X, X.T @ y); print('theta* =', theta_exact)

## 1. Descente de gradient
À COMPLÉTER : le gradient de L(θ) = (1/n)‖Xθ − y‖².

In [ ]:
def descente(X, y, eta, iters=300):
    theta, pertes = np.zeros(X.shape[1]), []
    for k in range(iters):
        grad = 2 / len(y) * X.T @ (X @ theta - y)
        theta = theta - eta * grad
        pertes.append(np.mean((X @ theta - y)**2))
    return theta, np.array(pertes)

In [ ]:
for eta in [0.05, 0.5, 0.77, 1.6]:
    th, L = descente(X, y, eta)
    plt.plot(np.clip(L, 1e-4, 1e3), label=f'eta={eta}')
    print(eta, th)
plt.yscale('log'); plt.legend(); plt.xlabel('itération'); plt.ylabel('perte'); plt.show()

## 2. Pourquoi η = 1,6 diverge-t-il ?

In [ ]:
H = 2 / n * X.T @ X
lam = np.linalg.eigvalsh(H)
print('valeurs propres de H :', lam)
print('eta max = 2 / lambda_max =', 2 / lam.max())
print('conditionnement =', lam.max() / lam.min())

## 3. Standardiser E/C

In [ ]:
Xs = np.c_[np.ones(n), (x - x.mean()) / x.std()]
print('valeurs propres de Hs :', np.linalg.eigvalsh(2 / n * Xs.T @ Xs))
th_s, Ls = descente(Xs, y, 0.5, iters=5)
print('après 5 itérations :', Ls)

## 4. Retrouver la loi d’Abrams
ln f_c = ln A − (E/C) ln B. À COMPLÉTER : A et B à partir de theta_exact.

In [ ]:
A = np.exp(theta_exact[0]); B = np.exp(-theta_exact[1])
print(f'f_c = {A:.0f} / {B:.2f}^(E/C)')